# Programming Assignment 1: Policy Learning for Recommendation

In this assignment you implement **policy-gradient (PG)** and **Plackett–Luce (PL)** ranking policies for a synthetic recommendation environment.


In [ ]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
%cd "/content/drive/MyDrive/{target_directory}"
!ls

## Libraries

We use PyTorch for all tensor operations ([tensor docs](https://docs.pytorch.org/docs/stable/tensors.html)).
Plotting uses `matplotlib` and `seaborn`.


In [ ]:
import torch
import torch.nn as nn
torch.__version__

'2.11.0+cu128'

In [ ]:
import time
from typing import Optional, Tuple, List
from abc import ABC, abstractmethod
from dataclasses import dataclass

from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
from base import BasePolicy, BaseEnv
from env import RecEnv

## 1. Plackett–Luce ranking (`pl.py`)

`PlackettLucePolicy` extends the two-tower model with:
- **Gumbel-top-$k$ sampling** for drawing ranked lists without backprop through the sampler
- **Sampling-with-replacement (SwR) log-prob** as a tractable approximation to the true PL likelihood
- **State-conditioned sampling** for the autoregressive trainer (`sample_action_given_state` / `calc_log_prob_given_state`)

Start on CPU; switch to GPU if training is too slow.


In [ ]:
from pl import (
    TwoTowerModel,
    PlackettLucePolicy,
    train_PLPolicy_efficiently,
    train_PLPolicy_autoregressively,
)

In [ ]:
# Visualization helpers live in visualize.py
# CLI:
#   python visualize.py -h
#   python visualize.py --alg PG --steps 2000
#   python visualize.py --alg PL --steps 2000

import importlib
import visualize
importlib.reload(visualize)

from visualize import (
    visualize_results,
    plot_reward_curve,
    plot_learned_policy,
    plot_item_ranking_distribution,
    run_pl_seed_sweep,
    visualize_seed_sweep,
)


### 1.1 Ranking-length sweep

`run_efficient_pg_across_K` trains efficient policy gradient for several list lengths $K \in \{1,3,5\}$. Uncomment the example call, then pass the results into `visualize_results(..., k_values=..., k_eval_values=..., k_compute_times=...)`.


In [ ]:
def run_efficient_pg_across_K(
    k_values: List[int] = [1, 3, 5],
    n_steps: int = 1000,
    batch_size: int = 32,
    n_users: int = 200,
    n_items: int = 400,
    n_dim: int = 10,
    model_dim: int = 14,
):
    """Train efficient PolicyGradient for several ranking lengths K.

    Uses a smaller env by default so the sweep finishes quickly for the assignment.
    Returns (k_values, final_eval_rewards, compute_times) for fig3.
    """
    final_evals = []
    times = []

    for K in k_values:
        env = RecEnv(n_users=n_users, n_items=n_items, n_dim=n_dim, reward_type="rating")
        policy = PlackettLucePolicy(
            n_users=n_users, n_items=n_items, model_dim=model_dim, reward_type="rating"
        )
        print(f"\n=== Efficient PG with ranking_length K={K} ===")
        _, _, eval_values, compute_time, *_ = train_PLPolicy_efficiently(
            env=env,
            policy=policy,
            loss_type="PolicyGradient",
            n_steps=n_steps,
            batch_size=batch_size,
            ranking_length=K,
        )
        final_evals.append(float(eval_values[-1]))
        times.append(float(compute_time))

    return list(k_values), final_evals, times


# Example:
# k_values, k_eval_values, k_compute_times = run_efficient_pg_across_K(k_values=[1, 3, 5], n_steps=1000)


### 1.2 Train three PL objectives

We train the same `PlackettLucePolicy` with three losses on a small `RecEnv` (`rating` reward):
1. **Regression** — MSE between `predict_value` and observed reward
2. **PolicyGradient (efficient)** — one-shot Gumbel-top-$k$ sample + SwR log-prob
3. **AutoRegressive** — step-by-step sampling through `env.reset()` / `env.step()`


In [ ]:
N_USERS = 200
N_ITEMS = 400
N_DIM = 10
MODEL_DIM = 14
N_STEPS = 1000

env_pl = RecEnv(n_users=N_USERS, n_items=N_ITEMS, n_dim=N_DIM, reward_type="rating")

trained_policy_reg, train_losses_reg, eval_values_reg, compute_time_reg, *_ = train_PLPolicy_efficiently(
    env=env_pl,
    policy=PlackettLucePolicy(n_users=N_USERS, n_items=N_ITEMS, model_dim=MODEL_DIM, reward_type="rating"),
    loss_type="Regression",
    n_steps=N_STEPS,
)


In [ ]:
trained_policy_pg, train_losses_pg, eval_values_pg, compute_time_pg, *_ = train_PLPolicy_efficiently(
    env=env_pl,
    policy=PlackettLucePolicy(n_users=N_USERS, n_items=N_ITEMS, model_dim=MODEL_DIM, reward_type="rating"),
    loss_type="PolicyGradient",
    n_steps=N_STEPS,
)


In [ ]:
trained_policy_ar, train_losses_ar, eval_values_ar, compute_time_ar, *_ = train_PLPolicy_autoregressively(
    env=RecEnv(n_users=N_USERS, n_items=N_ITEMS, n_dim=N_DIM, reward_type="rating"),
    policy=PlackettLucePolicy(n_users=N_USERS, n_items=N_ITEMS, model_dim=MODEL_DIM, reward_type="rating"),
    loss_type="AutoRegressive",
    n_steps=N_STEPS,
)


### 1.3 Visualize PL results


In [ ]:
visualize_results(
    train_losses=[train_losses_reg, train_losses_pg, train_losses_ar],
    eval_values=[eval_values_reg, eval_values_pg, eval_values_ar],
    compute_times=[compute_time_reg, compute_time_pg, compute_time_ar],
    loss_types=["Regression", "PolicyGradient", "AutoRegressive"],
    save_dir="figures",
)


### 1.4 Multi-seed stability check

Runs the PL trainers across several random seeds and plots mean ± std of final eval reward. Use this to argue whether PG consistently beats regression (or vice versa) under your hyperparameters.


In [ ]:
# Multi-seed sweep: Regression vs Policy Gradient
SEEDS = [0, 1, 2, 3, 4]

seed_results = run_pl_seed_sweep(
    seeds=SEEDS,
    n_users=N_USERS,
    n_items=N_ITEMS,
    n_dim=N_DIM,
    model_dim=MODEL_DIM,
    n_steps=N_STEPS,
    include_autoregressive=False,  # set True to also train autoregressive per seed (slower)
)

visualize_seed_sweep(seed_results, save_dir="figures")

## 2. Softmax policy (`pg.py`, $K = 1$)

`pg.py` is the single-item special case of the ranking problem. The policy is a **categorical softmax** over items.


In [ ]:
from pg import SoftmaxPolicy, train_SoftmaxPolicy

### 2.1 Train PG policies

Train REINFORCE and regression baselines on the same environment size, then run **§2.2** for plots.


In [ ]:
# ============================================================
# PG Policy Training Execution
# ============================================================

# Config where PG > supervised (mild over-param, rating reward)
N_USERS = 200
N_ITEMS = 400
N_DIM = 10
MODEL_DIM = 14
N_STEPS = 3000

print("Initializing Environment and Policies...")
env_pg = RecEnv(n_users=N_USERS, n_items=N_ITEMS, n_dim=N_DIM, reward_type="rating")

policy_reinforce = SoftmaxPolicy(n_users=N_USERS, n_items=N_ITEMS, model_dim=MODEL_DIM, reward_type="rating")
policy_variance = SoftmaxPolicy(n_users=N_USERS, n_items=N_ITEMS, model_dim=MODEL_DIM, reward_type="rating")

print("\n--- Training Standard REINFORCE ---")
_, losses_pg, evals_pg, time_pg, *_ = train_SoftmaxPolicy(
    env=env_pg,
    policy=policy_reinforce,
    loss_type="PolicyGradient",
    n_steps=N_STEPS,
    batch_size=128,
)
print(f"Final Eval Value: {evals_pg[-1]:.4f}")

print("\n--- Training Regression ---")
_, losses_var, evals_var, time_var, *_ = train_SoftmaxPolicy(
    env=env_pg,
    policy=policy_variance,
    loss_type="Regression",
    n_steps=N_STEPS,
    batch_size=128,
)
print(f"Final Eval Value: {evals_var[-1]:.4f}")


### 2.2 Visualize PG results

PG logs eval reward every 100 steps (not every step), so use `plot_reward_curve` with matching `eval_steps`.
The combined figure below compares REINFORCE vs regression on the same axes.


In [ ]:
import os

os.makedirs("figures", exist_ok=True)
EVAL_EVERY = 100
eval_steps = list(range(0, N_STEPS, EVAL_EVERY))[: len(evals_pg)]

plot_reward_curve(
    eval_steps,
    evals_pg,
    losses_pg,
    title="PG — REINFORCE",
    save_path="figures/pg_reinforce_curve.png",
)
plot_reward_curve(
    eval_steps,
    evals_var,
    losses_var,
    title="PG — Regression",
    save_path="figures/pg_regression_curve.png",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(eval_steps, evals_pg.detach().cpu().numpy(), label="REINFORCE", color="#d62728", linewidth=1.8)
axes[0].plot(eval_steps, evals_var.detach().cpu().numpy(), label="Regression", color="#1f77b4", linewidth=1.8)
axes[0].set_title("Eval reward: REINFORCE vs Regression")
axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Expected reward")
axes[0].legend()

axes[1].plot(losses_pg.detach().cpu().numpy(), label="REINFORCE", color="#d62728", alpha=0.85)
axes[1].plot(losses_var.detach().cpu().numpy(), label="Regression", color="#1f77b4", alpha=0.85)
axes[1].set_title("Training loss")
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Loss")
axes[1].legend()
fig.tight_layout()
fig.savefig("figures/pg_reinforce_vs_regression.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/pg_reinforce_vs_regression.png")

plot_learned_policy(
    policy_reinforce,
    save_path="figures/pg_reinforce_policy.png",
    title="REINFORCE learned policy",
)
plot_learned_policy(
    policy_variance,
    save_path="figures/pg_regression_policy.png",
    title="Regression learned policy",
)
